# Bài 4.2: Xây dựng Hệ thống Agentic RAG Tiếng Việt với `LangGraph` & Hybrid Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/04_Vietnamese_LLMs_and_Agents/02_vietnamese_rag_langgraph_pipeline.ipynb)

> **Mục tiêu bài học:**
> 1. Thiết kế bộ truy xuất lai (**Hybrid Retriever**) tối ưu cho tiếng Việt: kết hợp **BM25 có tách từ tiếng Việt** (bắt chính xác tên riêng, mã luật, thuật ngữ) + **Dense Vector Embedding** (`BAAI/bge-m3` hoặc `hiieu/halong_embedding`).
> 2. Xây dựng luồng **Agentic Self-RAG** bằng **LangGraph** trên dữ liệu **`UIT-ViQuAD 2.0`**:
>    - `retrieve`: Truy xuất tài liệu từ Hybrid Index.
>    - `grade_documents`: Đánh giá mức độ liên quan của tài liệu (chống ảo giác).
>    - `rewrite_query`: Tự động viết lại câu hỏi nếu tài liệu truy xuất chưa đạt yêu cầu.
>    - `generate`: Sinh câu trả lời có trích dẫn nguồn bằng LLM (Local Qwen hoặc Gemini API).

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../01_Foundations_Preprocessing").resolve()))
from vn_preprocessing import clean_vietnamese_text

# Đọc dữ liệu mẫu UIT-ViQuAD 2.0
viquad_path = Path("../datasets/samples/viquad_qa_sample.jsonl")
with open(viquad_path, "r", encoding="utf-8") as f:
    qa_samples = [json.loads(line) for line in f]

for item in qa_samples:
    print(f"[{item['id']}] Chủ đề  : {item['title']}")
    print(f"          Câu hỏi : {item['question']}")
    print(f"          Đáp án  : {item['answers']['text'][0]}\n")

[viquad_001] Chủ đề  : Xử lý ngôn ngữ tự nhiên tiếng Việt
          Câu hỏi : Tại sao khoảng trắng trong tiếng Việt không trực tiếp phân tách từ như tiếng Anh?
          Đáp án  : Khoảng trắng trong tiếng Việt dùng để phân tách các âm tiết (hay còn gọi là tiếng)

[viquad_002] Chủ đề  : Mô hình PhoBERT
          Câu hỏi : Dữ liệu đầu vào của mô hình PhoBERT cần được tiền xử lý bằng công cụ nào trước khi đưa vào tokenizer?
          Đáp án  : RDRSegmenter (VnCoreNLP)


In [2]:
# Minh họa kiến trúc StateGraph của Agentic RAG Tiếng Việt (LangGraph)
from typing import TypedDict


class AgenticRAGState(TypedDict):
    question: str
    rewritten_question: str
    documents: list[str]
    is_relevant: bool
    answer: str


def retrieve_node(state: AgenticRAGState) -> dict:
    query = state.get("rewritten_question") or state["question"]
    cleaned_q = clean_vietnamese_text(query, lowercase=True)
    # Tìm kiếm đơn giản trên tập mẫu ViQuAD
    matched = [
        row["context"]
        for row in qa_samples
        if any(tok in row["context"].lower() for tok in cleaned_q.split())
    ]
    return {"documents": matched}


def grade_documents_node(state: AgenticRAGState) -> dict:
    docs = state.get("documents", [])
    return {"is_relevant": len(docs) > 0}


def generate_answer_node(state: AgenticRAGState) -> dict:
    context = "\n".join(state["documents"])
    return {
        "answer": f"[Trích xuất từ ngữ cảnh ViQuAD]: {context[:210]}..."
    }


# Chạy thử luồng State
initial_state: AgenticRAGState = {
    "question": "PhoBERT cần tiền xử lý bằng công cụ tách từ nào?",
    "rewritten_question": "",
    "documents": [],
    "is_relevant": False,
    "answer": "",
}
state_after_retrieve = {**initial_state, **retrieve_node(initial_state)}
state_after_grade = {**state_after_retrieve, **grade_documents_node(state_after_retrieve)}
final_state = {**state_after_grade, **generate_answer_node(state_after_grade)}

print("Câu hỏi :", final_state["question"])
print("Relevant:", final_state["is_relevant"])
print("Trả lời :", final_state["answer"])

Câu hỏi : PhoBERT cần tiền xử lý bằng công cụ tách từ nào?
Relevant: True
Trả lời : [Trích xuất từ ngữ cảnh ViQuAD]: Tiếng Việt là một ngôn ngữ đơn lập và phân tích tính, trong đó ranh giới giữa các từ không được phân tách tự nhiên bằng khoảng trắng. Khoảng trắng trong tiếng Việt dùng để phân tách các âm tiết (hay còn gọi là ...
